# Trabajo Práctico Data Warehouse — Siniestros Viales (CABA)

**Tema:** Análisis multidimensional de siniestros viales en CABA..

Fuente: [BA Data - Siniestros viales](https://data.buenosaires.gob.ar/dataset/victimas-siniestros-viales)

**Alumnos**: Gabriel Acebal, Inca Julian Marcelo Sayanco Navarro, María Delfina Torres, Tomas Orlando y Felipe Gimenez - (Grupo 1)

**Profesores**: Gonzalo Javier Díaz y Julio Paredes Rojas


Este notebook arma el modelo dimensional (1 tabla de hechos + 3 dimensiones) a partir de los dos archivos fuente:
- `hechos.csv` (un registro por siniestro: fecha, hora, ubicación, tipo de vía, vehículos involucrados)
- `victimas.csv` (un registro por persona afectada: sexo, edad, rol, gravedad)

## 1. Carga de datos

Descargá los dos archivos del portal de datos abiertos, subilos a tu Google Drive y cambiá el path donde se encuentren antes de correr esta celda:
- Hechos (CSV): https://data.buenosaires.gob.ar/dataset/victimas-siniestros-viales/resource/40ec993a-00ad-40e5-936f-0a25f8d2c90b/download
- Víctimas (CSV): https://data.buenosaires.gob.ar/dataset/victimas-siniestros-viales/resource/41beafc0-ca1c-430d-a29d-07c425d0aa20/download

In [48]:
# Importamos las herramientas necesarias:
# - drive: conecta Google Colab con Google Drive.
# - Path: permite trabajar con rutas de archivos y carpetas.
# - pandas: permite cargar y manipular datos en tablas (DataFrames).
from google.colab import drive
from pathlib import Path
import pandas as pd

# Montamos Google Drive para acceder a sus archivos desde Colab.
# force_remount=True vuelve a montarlo aunque ya estuviera conectado.
drive.mount("/content/drive", force_remount=True)

# Definimos la carpeta principal de búsqueda: "Mi unidad" de Google Drive.
base_path = Path("/content/drive/MyDrive")


# Busca un archivo por su nombre dentro de Mi unidad y sus subcarpetas.
# Devuelve su ruta si encuentra exactamente una coincidencia.
def buscar_archivo(nombre):
    # Recorremos las subcarpetas y reunimos las rutas que coinciden.
    coincidencias = list(base_path.rglob(nombre))

    # Si el archivo no existe, detenemos la ejecución con un mensaje.
    if not coincidencias:
        raise FileNotFoundError(
            f"No se encontró {nombre} dentro de Mi unidad. "
            "Subí el CSV a tu Google Drive y ejecutá nuevamente."
        )

    # Si hay varias copias, mostramos sus rutas y detenemos la ejecución
    # para evitar seleccionar automáticamente un archivo incorrecto.
    if len(coincidencias) > 1:
        # Convertimos las rutas a texto y las separamos con saltos de línea.
        rutas = "\n".join(str(p) for p in coincidencias)
        raise ValueError(
            f"Hay varias copias de {nombre}. "
            f"Debés elegir cuál usar:\n{rutas}"
        )

    # Devolvemos la ruta del único archivo encontrado.
    return coincidencias[0]


# Buscamos los CSV de siniestros y víctimas.
hechos_path = buscar_archivo("siniestros_viales_hechos.csv")
victimas_path = buscar_archivo("siniestros_viales_victimas.csv")

# Mostramos las rutas para verificar qué archivos se cargarán.
print("Ruta de Hechos:", hechos_path)
print("Ruta de Víctimas:", victimas_path)

# Cargamos el CSV de hechos en un DataFrame.
# sep=";": los campos están separados por punto y coma.
# encoding="utf-8-sig": interpreta UTF-8 y elimina el BOM inicial si existe.
# low_memory=False: evita inferir tipos por bloques internos de lectura;
# puede consumir más memoria, pero reduce las inferencias inconsistentes.
df_hechos = pd.read_csv(
    hechos_path,
    sep=";",
    encoding="utf-8-sig",
    low_memory=False
)

# Cargamos el CSV de víctimas utilizando la misma configuración.
df_victimas = pd.read_csv(
    victimas_path,
    sep=";",
    encoding="utf-8-sig",
    low_memory=False
)

# Normalizamos los nombres de las columnas de ambos DataFrames:
# strip() elimina espacios al principio y al final.
# lower() convierte los nombres a minúsculas.
df_hechos.columns = df_hechos.columns.str.strip().str.lower()
df_victimas.columns = df_victimas.columns.str.strip().str.lower()

# Mostramos los nombres de las columnas para revisar la estructura.
print("Columnas de Hechos:", df_hechos.columns.tolist())
print("Columnas de Víctimas:", df_victimas.columns.tolist())

# Mostramos la cantidad de filas cargadas en cada DataFrame.
print("Registros Hechos:", len(df_hechos))
print("Registros Víctimas:", len(df_victimas))

Mounted at /content/drive
Ruta de Hechos: /content/drive/MyDrive/UP/MATERIAS/2026 (4)/2do Semestre/Analisis de la Informacion y la Decision /07.1 TP (Data Warehouse) AID/DataSets/Hechos/siniestros_viales_hechos.csv
Ruta de Víctimas: /content/drive/MyDrive/UP/MATERIAS/2026 (4)/2do Semestre/Analisis de la Informacion y la Decision /07.1 TP (Data Warehouse) AID/DataSets/Victimas/siniestros_viales_victimas.csv
Columnas de Hechos: ['id_siniestro', 'numero_total_de_victimas', 'numero_victimas_leve_siniestro', 'numero_victimas_grave_siniestro', 'numero_victimas_mortal_siniestro', 'fecha_siniestro', 'anio_siniestro', 'mes_siniestro', 'dia_siniestro', 'hora_siniestro', 'rango_horario', 'direccion_normalizada_siniestro', 'comuna_siniestro', 'tipo_de_via_siniestro', 'geocodificacion_plana', 'longitud_siniestro', 'latitud_siniestro', 'participantes_siniestro', 'modo_desplazamiento_victima', 'contraparte_siniestro', 'gravedad_siniestro']
Columnas de Víctimas: ['id_siniestro', 'fecha_siniestro', 'an

## 2. Dimensión Tiempo

Jerarquía: **Año > Mes > Día > Hora** (con el atributo adicional Turno).

El archivo de hechos ya trae `aaaa`, `mm`, `dd` y `hh` como columnas separadas, así que armamos el id numérico directamente sin necesidad de parsear fechas.

In [49]:
# Clasificamos la hora del siniestro en una franja horaria.
# hh representa la hora del día, con valores de 0 a 23.
def calcular_turno(hh):
    # Si la hora está ausente, indicamos que no hay información.
    if pd.isna(hh):
        return "Sin dato"

    # Asignamos el turno según el intervalo al que pertenece la hora.
    if 0 <= hh < 6:
        return "Madrugada"  # Desde las 00:00 hasta las 05:59.
    elif 6 <= hh < 12:
        return "Mañana"     # Desde las 06:00 hasta las 11:59.
    elif 12 <= hh < 19:
        return "Tarde"      # Desde las 12:00 hasta las 18:59.
    else:
        return "Noche"      # Desde las 19:00 hasta las 23:59.


# Convertimos la columna de fechas a valores de tipo datetime.
# format="%Y-%m-%d": indica el formato año-mes-día.
# errors="coerce": convierte los valores inválidos en NaT (fecha ausente).
fechas = pd.to_datetime(
    df_hechos["fecha_siniestro"],
    format="%Y-%m-%d",
    errors="coerce"
)

# Interpretamos la hora en formato horas:minutos:segundos.
# dt.hour extrae únicamente la hora, entre 0 y 23.
# Int64 permite almacenar números enteros y valores ausentes (pd.NA).
horas = pd.to_datetime(
    df_hechos["hora_siniestro"],
    format="%H:%M:%S",
    errors="coerce"
).dt.hour.astype("Int64")

# Si existe alguna fecha vacía o inválida, detenemos la ejecución.
# Necesitamos una fecha válida para construir el identificador temporal.
if fechas.isna().any():
    raise ValueError("Hay fechas vacías o inválidas; revisalas antes de continuar.")

# Creamos el DataFrame de la dimensión tiempo con el mismo índice
# que df_hechos para mantener la correspondencia entre las filas.
dim_tiempo = pd.DataFrame(index=df_hechos.index)

# Extraemos los componentes de la fecha para facilitar el análisis
# y la agrupación de los siniestros por año, mes y día.
dim_tiempo["anio"] = fechas.dt.year.astype("Int64")
dim_tiempo["mes"] = fechas.dt.month.astype("Int64")
dim_tiempo["dia"] = fechas.dt.day.astype("Int64")

# Incorporamos la hora y calculamos su turno mediante la función definida.
dim_tiempo["hora"] = horas
dim_tiempo["turno"] = horas.apply(calcular_turno)

# Construimos una clave temporal con el formato numérico AAAAMMDDHH.
# Ejemplo: 2026091314 representa el 13/09/2026 a las 14 horas.
# Si la hora es desconocida, usamos 99 únicamente para construir la clave;
# la columna "hora" conserva su valor ausente.
dim_tiempo["id_tiempo"] = (
    dim_tiempo["anio"] * 1_000_000
    + dim_tiempo["mes"] * 10_000
    + dim_tiempo["dia"] * 100
    + dim_tiempo["hora"].fillna(99)
).astype("Int64")

# Organizamos la dimensión tiempo:
# - Seleccionamos las columnas en el orden deseado.
# - Dejamos una sola fila por cada combinación de fecha y hora.
# - Ordenamos las filas por su identificador temporal.
# - Reiniciamos el índice sin conservar el índice anterior como columna.
dim_tiempo = (
    dim_tiempo[
        ["id_tiempo", "anio", "mes", "dia", "hora", "turno"]
    ]
    .drop_duplicates(subset=["id_tiempo"])
    .sort_values("id_tiempo")
    .reset_index(drop=True)
)

# Mostramos las primeras cinco filas para revisar el resultado.
print(dim_tiempo.head())

# Mostramos la cantidad de registros únicos de la dimensión tiempo.
print("Filas en Dim_Tiempo:", len(dim_tiempo))

    id_tiempo  anio  mes  dia  hora      turno
0  2019010101  2019    1    1     1  Madrugada
1  2019010102  2019    1    1     2  Madrugada
2  2019010103  2019    1    1     3  Madrugada
3  2019010104  2019    1    1     4  Madrugada
4  2019010107  2019    1    1     7     Mañana
Filas en Dim_Tiempo: 34735


## 3. Dimensión Ubicación

Jerarquía: **Comuna > Tipo de calle > Calle**.

A diferencia de Ecobici (donde cada estación tiene un ID fijo en un catálogo), acá no existe un catálogo maestro de ubicaciones: cada siniestro trae su propia comuna/calle como texto libre. Por eso generamos un ID sustituto (surrogate key) a partir de las combinaciones únicas que aparecen en los datos.

In [50]:
# Definimos las columnas de origen y los nombres que tendrán
# dentro de la dimensión ubicación.
columnas_ubicacion = {
    "comuna_siniestro": "comuna",
    "tipo_de_via_siniestro": "tipo_de_calle",
    "direccion_normalizada_siniestro": "direccion"
}

# Creamos la dimensión ubicación a partir de los datos de siniestros.
dim_ubicacion = (
    # Seleccionamos las columnas usando las claves del diccionario.
    df_hechos[list(columnas_ubicacion)]

    # Renombramos las columnas según el mapeo definido.
    .rename(columns=columnas_ubicacion)

    # Conservamos una sola fila por cada combinación
    # de comuna, tipo de calle y dirección.
    .drop_duplicates()

    # Reiniciamos el índice desde cero, sin guardar el índice anterior.
    .reset_index(drop=True)
)

# Agregamos "id_ubicacion" como primera columna (posición 0).
# Asignamos un identificador consecutivo desde 1 a cada ubicación.
# Esta clave permitirá relacionar la dimensión con la tabla de hechos.
dim_ubicacion.insert(
    0, "id_ubicacion", dim_ubicacion.index + 1
)

# Mostramos las primeras cinco filas para revisar el resultado.
print(dim_ubicacion.head())

# Mostramos la cantidad de ubicaciones únicas de la dimensión.
print("Filas en Dim_Ubicacion:", len(dim_ubicacion))

   id_ubicacion     comuna tipo_de_calle                           direccion
0             1   Comuna 8     AUTOPISTA                                 NaN
1             2  Comuna 12       AVENIDA  DE LOS CONSTITUYENTES AV. y HABANA
2             3  Comuna 11           NaN                                 NaN
3             4   Comuna 9           NaN                                 NaN
4             5   Comuna 4           NaN                                 NaN
Filas en Dim_Ubicacion: 30936


## 4. Dimensión Víctima

Jerarquía: **Rango etario > Edad**. Atributos adicionales: Sexo, Rol de la víctima, Tipo de vehículo.

El dataset no tiene un ID persistente de víctima (una misma persona no se puede rastrear entre siniestros), así que modelamos esta dimensión como una **dimensión de perfil deduplicada**: cada combinación única de edad/sexo/rol/vehículo se guarda una sola vez.

In [51]:
# Convertimos la edad de la víctima de texto a un valor numérico.
# errors="coerce" transforma "SD" (sin dato) y otros valores
# no numéricos en NaN (valor ausente).
df_victimas["edad_numerica"] = pd.to_numeric(
    df_victimas["edad_victima"], errors="coerce"
)

# Consideramos válidas las edades entre 0 y 120 años, inclusive.
# where() conserva las edades válidas y reemplaza las demás por NaN.
# El resultado se guarda en "edades", sin modificar "edad_numerica".
edades = df_victimas["edad_numerica"].where(
    df_victimas["edad_numerica"].between(0, 120)
)

# Definimos los límites de los intervalos para agrupar las edades.
# Cada intervalo incluirá el límite izquierdo y excluirá el derecho.
# Por ejemplo, [18, 26) agrupa las edades desde 18 hasta antes de 26.
bins = [0, 18, 26, 36, 46, 61, 121]

# Definimos una etiqueta por cada intervalo.
etiquetas = [
    "Menores de 18",
    "18-25",
    "26-35",
    "36-45",
    "46-60",
    "Mayores de 60"
]

# Creamos el rango etario de cada víctima:
# - pd.cut() clasifica las edades en los intervalos definidos.
# - right=False excluye el extremo derecho de cada intervalo.
# - add_categories() incorpora "Sin dato" como categoría permitida.
# - fillna() asigna esa categoría a las edades ausentes o inválidas.
df_victimas["rango_etario"] = (
    pd.cut(edades, bins=bins, labels=etiquetas, right=False)
    .cat.add_categories(["Sin dato"])
    .fillna("Sin dato")
)

# Seleccionamos los atributos que definen un perfil de víctima.
# La dimensión representa perfiles, no personas individuales.
cols_victima = [
    "rango_etario",
    "sexo_victima",
    "rol_victima",
    "modo_desplazamiento_victima"
]

# Creamos la dimensión de perfiles de víctimas.
dim_victima = (
    # Seleccionamos los atributos definidos anteriormente.
    df_victimas[cols_victima]

    # Conservamos una sola fila por cada combinación de rango etario,
    # sexo, rol y modo de desplazamiento.
    .drop_duplicates()

    # Reiniciamos el índice desde cero, sin guardar el índice anterior.
    .reset_index(drop=True)
)

# Insertamos una clave consecutiva desde 1 como primera columna.
# Permitirá relacionar cada registro de víctima con su perfil
# desde la tabla de hechos del Data Warehouse.
dim_victima.insert(
    0, "id_victima_perfil", dim_victima.index + 1
)

# Mostramos las primeras cinco filas para revisar el resultado.
print(dim_victima.head())

# Mostramos la cantidad de perfiles únicos de la dimensión.
print("Filas en Dim_Victima:", len(dim_victima))

   id_victima_perfil   rango_etario sexo_victima rol_victima  \
0                  1          18-25            F    CICLISTA   
1                  2  Menores de 18            F    CICLISTA   
2                  3          26-35            F    CICLISTA   
3                  4          36-45            F    CICLISTA   
4                  5          46-60            F    CICLISTA   

  modo_desplazamiento_victima  
0                   BICICLETA  
1                   BICICLETA  
2                   BICICLETA  
3                   BICICLETA  
4                   BICICLETA  
Filas en Dim_Victima: 357


## 5. Tabla de Hechos: Fact_Siniestros

**Grano:** una fila = una víctima involucrada en un siniestro (join de `victimas` con `hechos` por `id_hecho`).

**Medidas:**
- `cantidad_victimas` (implícita, 1 por registro)
- `es_fallecido` (1 si `gravedad == 'MORTAL'`, 0 en caso contrario)

**Claves foráneas:** `id_tiempo`, `id_ubicacion`, `id_victima_perfil` + el identificador degenerado `id_hecho` (para trazabilidad al siniestro original).

In [52]:
# 1. Calculamos la clave temporal de cada siniestro usando
# la misma regla aplicada al construir Dim_Tiempo.

# Convertimos las fechas al formato datetime.
# Los valores vacíos o inválidos se convierten en NaT.
fechas = pd.to_datetime(
    df_hechos["fecha_siniestro"],
    format="%Y-%m-%d",
    errors="coerce"
)

# Interpretamos las horas y extraemos su componente horario (0 a 23).
# Int64 permite almacenar números enteros y valores ausentes.
horas = pd.to_datetime(
    df_hechos["hora_siniestro"],
    format="%H:%M:%S",
    errors="coerce"
).dt.hour.astype("Int64")

# Detenemos la ejecución si hay fechas ausentes o inválidas,
# porque no podemos construir una clave temporal válida.
if fechas.isna().any():
    raise ValueError("Hay fechas vacías o inválidas en df_hechos.")

# Construimos id_tiempo con el formato numérico AAAAMMDDHH.
# Usamos 99 cuando la hora es desconocida, igual que en Dim_Tiempo.
df_hechos["id_tiempo"] = (
    fechas.dt.year * 1_000_000
    + fechas.dt.month * 10_000
    + fechas.dt.day * 100
    + horas.fillna(99)
).astype("Int64")


# 2. Asociamos cada siniestro con su ubicación para obtener id_ubicacion.
hechos_con_claves = (
    # Eliminamos una clave previa para evitar columnas duplicadas
    # al volver a ejecutar la celda.
    # errors="ignore" evita un error si la columna no existe.
    df_hechos.drop(columns=["id_ubicacion"], errors="ignore")
    .merge(
        dim_ubicacion,

        # Columnas de ubicación presentes en los datos de siniestros.
        left_on=[
            "comuna_siniestro",
            "tipo_de_via_siniestro",
            "direccion_normalizada_siniestro"
        ],

        # Columnas equivalentes dentro de la dimensión ubicación.
        right_on=["comuna", "tipo_de_calle", "direccion"],

        # Conservamos todos los siniestros, incluso sin coincidencia.
        # Si no encontramos una ubicación, su clave queda ausente.
        how="left",

        # Varios siniestros pueden compartir una ubicación,
        # pero la combinación debe ser única en dim_ubicacion.
        validate="many_to_one"
    )
)


# 3. Asociamos cada registro de víctima con su perfil.

# Definimos los atributos compartidos entre los datos de víctimas
# y la dimensión de perfiles.
cols_victima = [
    "rango_etario",
    "sexo_victima",
    "rol_victima",
    "modo_desplazamiento_victima"
]

victimas_con_claves = (
    # Eliminamos claves previas para obtenerlas nuevamente
    # sin generar columnas duplicadas durante las uniones.
    df_victimas.drop(
        columns=["id_victima_perfil", "id_tiempo", "id_ubicacion"],
        errors="ignore"
    )
    .merge(
        dim_victima,

        # Unimos por la combinación de atributos del perfil.
        on=cols_victima,

        # Conservamos todas las víctimas, aunque no tengan un perfil asociado.
        how="left",

        # Varias víctimas pueden compartir un perfil,
        # pero cada combinación debe ser única en dim_victima.
        validate="many_to_one"
    )
)


# 4. Construimos la tabla de hechos con una fila por registro de víctima.
# Incorporamos las claves de tiempo y ubicación del siniestro asociado.
fact_siniestros = victimas_con_claves.merge(
    hechos_con_claves[
        ["id_siniestro", "id_tiempo", "id_ubicacion"]
    ],

    # Vinculamos cada víctima con su siniestro.
    on="id_siniestro",

    # Conservamos también las víctimas sin un siniestro asociado.
    how="left",

    # Un siniestro puede tener varias víctimas, pero su identificador
    # debe aparecer una sola vez en hechos_con_claves.
    validate="many_to_one"
)

# Cada fila representa un registro de víctima y aporta 1 al conteo.
# Al sumar esta medida obtenemos la cantidad de registros de víctimas.
fact_siniestros["cantidad_victimas"] = 1

# Creamos un indicador numérico de fallecimiento:
# - Convertimos la gravedad a texto.
# - Quitamos espacios al principio y al final.
# - Convertimos a mayúsculas para uniformar la comparación.
# - Asignamos 1 si el valor es "MORTAL" y 0 en los demás casos.
# Los valores ausentes también se codifican como 0; esto no confirma
# que la víctima haya sobrevivido, sino que no figura como "MORTAL".
fact_siniestros["es_fallecido"] = (
    fact_siniestros["gravedad_victima"]
    .astype("string")
    .str.strip()
    .str.upper()
    .eq("MORTAL")
    .fillna(False)
    .astype(int)
)


# 5. Separamos los registros que tienen alguna clave dimensional ausente.

# Definimos las claves necesarias para relacionar la tabla de hechos
# con las dimensiones de tiempo, ubicación y perfil de víctima.
claves = ["id_tiempo", "id_ubicacion", "id_victima_perfil"]

# isna() identifica valores ausentes.
# any(axis=1) marca True si falta al menos una clave en esa fila.
# Este control detecta claves ausentes, pero no comprueba que cada
# id_tiempo exista efectivamente dentro de Dim_Tiempo.
sin_claves = fact_siniestros[claves].isna().any(axis=1)

# Conservamos los registros incompletos, con todas sus columnas,
# en un DataFrame separado para revisarlos.
fact_siniestros_pendientes = fact_siniestros.loc[sin_claves].copy()

# Definimos las columnas finales de la tabla de hechos:
# identificador del siniestro, claves dimensionales y medidas.
# id_siniestro puede repetirse cuando un siniestro tiene varias víctimas.
columnas_fact = [
    "id_siniestro",
    "id_tiempo",
    "id_ubicacion",
    "id_victima_perfil",
    "cantidad_victimas",
    "es_fallecido"
]

# Conservamos únicamente las filas que tienen todas las claves.
# ~ invierte la condición: selecciona las filas sin claves ausentes.
# Creamos una copia y reiniciamos el índice desde cero.
fact_siniestros = (
    fact_siniestros.loc[~sin_claves, columnas_fact]
    .copy()
    .reset_index(drop=True)
)

# Aseguramos que las claves dimensionales tengan tipo entero Int64.
fact_siniestros[claves] = fact_siniestros[claves].astype("Int64")

# Mostramos las primeras cinco filas de la tabla de hechos.
print(fact_siniestros.head())

# Informamos cuántas filas quedaron listas y cuántas requieren revisión.
print("Filas en Fact_Siniestros:", len(fact_siniestros))
print("Registros pendientes de revisión:", len(fact_siniestros_pendientes))

      id_siniestro   id_tiempo  id_ubicacion  id_victima_perfil  \
0  LC-2019-0022650  2019011114            28                  1   
1  LC-2019-0068291  2019020115            42                  2   
2  LC-2019-0139186  2019030620           122                  3   
3  LC-2019-0247839  2019042117            12                  3   
4  LC-2019-0283677  2019050712            14                  4   

   cantidad_victimas  es_fallecido  
0                  1             0  
1                  1             0  
2                  1             0  
3                  1             0  
4                  1             0  
Filas en Fact_Siniestros: 75193
Registros pendientes de revisión: 4


## 6. Guardado de los archivos del Data Warehouse

In [53]:
# Importamos Path para trabajar con rutas de archivos y carpetas.
from pathlib import Path

# Definimos la carpeta de Google Drive donde guardaremos los CSV.
# Los espacios y paréntesis son válidos dentro de la ruta.
ruta_guardado = Path("/content/drive/MyDrive/UP/MATERIAS/2026 (4)/2do Semestre/Analisis de la Informacion y la Decision /07.1 TP (Data Warehouse) AID/TPDataWarehouseSiniestros")

# Creamos la carpeta de destino si todavía no existe.
# parents=True: crea también las carpetas intermedias necesarias.
# exist_ok=True: evita un error si la carpeta ya existe.
ruta_guardado.mkdir(parents=True, exist_ok=True)

# Asociamos cada nombre de archivo con el DataFrame que exportaremos.
# Incluimos la tabla de hechos y las tres dimensiones del Data Warehouse.
tablas = {
    "DW_Fact_Siniestros.csv": fact_siniestros,
    "DW_Dim_Tiempo.csv": dim_tiempo,
    "DW_Dim_Ubicacion.csv": dim_ubicacion,
    "DW_Dim_Victima.csv": dim_victima
}

# Recorremos el diccionario para guardar cada tabla en su archivo.
# nombre contiene el nombre del CSV y tabla contiene el DataFrame.
for nombre, tabla in tablas.items():
    # Exportamos el DataFrame a CSV.
    # Si ya existe un archivo con el mismo nombre, se reemplaza.
    tabla.to_csv(
        # Unimos la carpeta de destino con el nombre del archivo.
        ruta_guardado / nombre,

        # Excluimos el índice del DataFrame del archivo generado.
        index=False,

        # Usamos punto y coma como separador de columnas.
        sep=";",

        # Guardamos en UTF-8 con BOM para facilitar la lectura
        # de acentos y otros caracteres en programas como Excel.
        encoding="utf-8-sig"
    )

# Mostramos la carpeta donde se guardaron los cuatro archivos.
print("Archivos guardados en:", ruta_guardado)

Archivos guardados en: /content/drive/MyDrive/UP/MATERIAS/2026 (4)/2do Semestre/Analisis de la Informacion y la Decision /07.1 TP (Data Warehouse) AID/TPDataWarehouseSiniestros
